<a href="https://colab.research.google.com/github/cintiapinho/Aulas_FATEC_PLN/blob/main/Aulas_FATEC_PLN/Materiais/03_Processamento-de-texto-I%20e%20II/notebook_pratica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sessões 03 e 04 — Prática: tokenização, regex, stopwords, stemming e lematização

**Processamento de Linguagem Natural**

Este é o notebook de **prática guiada** — passo a passo, com exemplos já prontos pra você rodar e entender cada técnica. Os exercícios que você entrega ficam no notebook separado `notebook_exercicios.ipynb` — ele reaproveita tudo que é construído aqui.

In [ ]:
import nltk
import re
from collections import Counter

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

## 1. Tokenização, mais a fundo

Vamos tokenizar uma frase cheia de casos difíceis de propósito: contração, palavra composta, preço e URL.

In [ ]:
from nltk.tokenize import word_tokenize

frase_dificil = "Vi no site https://loja.com que o guarda-chuva tá R$ 29,90 — não é caro pelo que é."

tokens = word_tokenize(frase_dificil, language='portuguese')
print(tokens)

Repare o que aconteceu com a URL, com "guarda-chuva" e com "R$ 29,90". Antes de rodar a próxima célula, escreva (numa célula de markdown, mais abaixo) o que você esperava que acontecesse com cada um desses três — e se bateu.

## 2. Expressões regulares (regex) para limpar antes de tokenizar

Em vez de deixar o tokenizador lidar com a URL, podemos removê-la (ou qualquer outro padrão) antes, com regex.

In [ ]:
texto = frase_dificil

# Remove URLs (http ou https, seguido de qualquer coisa sem espaço)
texto_sem_url = re.sub(r'http\S+', '', texto)
print("Sem URL:", texto_sem_url)

# Remove tudo que não é letra, número ou espaço (pontuação, símbolos, mas mantém acentos)
texto_sem_pontuacao = re.sub(r'[^\w\s]', '', texto_sem_url)
print("Sem pontuação:", texto_sem_pontuacao)

# Normaliza espaços duplos que sobraram
texto_limpo = re.sub(r'\s+', ' ', texto_sem_pontuacao).strip()
print("Espaços normalizados:", texto_limpo)

Três linhas, três decisões diferentes. Se você mudar a ordem (por exemplo, tirar pontuação antes de tirar a URL), o resultado muda — teste se quiser.

## 3. Filtrando pontuação depois de tokenizar

A outra forma de resolver o mesmo problema: tokenizar normal, e filtrar os tokens que não são palavras.

In [ ]:
tokens = word_tokenize(frase_dificil, language='portuguese')
tokens_sem_pontuacao = [t for t in tokens if t.isalpha()]

print("Antes:", tokens)
print("Depois:", tokens_sem_pontuacao)

Repare que `.isalpha()` também derrubou o "29" e o "90" (números não são letras) e separou "guarda" de "chuva" (o hífen já tinha virado um token à parte na tokenização). Nenhuma das duas abordagens (regex antes, filtro depois) é "a certa" — dependem do que você precisa manter.

### Normalização: tirando acento

Antes de seguir pro regex, um recurso rápido que você vai usar hoje: quando a decisão for tirar acento, o módulo `unicodedata` (da biblioteca padrão do Python, não precisa instalar nada) resolve sem listar "á → a", "ç → c" etc. na mão.

Entrada: "Coração"

Normalização (NFKD): "C", "o", "r", "a", "c", "¸", "a", "~", "o"

Encode ASCII (ignore): "C", "o", "r", "a", "c", "a", "o" (o ¸ e o ~ foram jogados fora)

Resultado: "Coracao"

In [ ]:
import unicodedata

def remover_acentos(texto):
    # NFKD separa a letra do sinal de acento — "á" vira dois caracteres Unicode:
    # "a" + o acento agudo sozinho
    texto_normalizado = unicodedata.normalize('NFKD', texto)
    # .encode('ascii', 'ignore') descarta tudo que não é ASCII — inclusive o acento que sobrou
    return texto_normalizado.encode('ascii', 'ignore').decode('ascii')

print(remover_acentos("Processamento de Linguagem Natural é incrível! São Paulo, Açaí."))

O mesmo truque tira cedilha também ("ç" se separa em "c" + sinal) — serve pra qualquer acentuação do português numa linha só. Guarde essa função: ela reaparece no exercício de hoje.

Só tirar acento não deixa o texto pronto pra virar nome de arquivo ou identificador — ainda sobra espaço e pontuação. O mesmo `re.sub` resolve, combinando uma classe negada (`[^...]`, "tudo que não é isso") com `+` (uma ou mais vezes) — assim, uma sequência inteira de espaço/pontuação vira um separador só, não um por caractere:

In [ ]:
texto = "Vendas  do 1º trimestre - 2026!!"

# [^a-z0-9]+ = "uma ou mais vezes, qualquer coisa que não seja letra minúscula nem número"
texto_limpo = re.sub(r'[^a-z0-9]+', '_', texto.lower())
texto_limpo = texto_limpo.strip('_')  # tira "_" que sobrou no início/fim

print(texto_limpo)

Repare: "º" também sumiu — não é letra nem número, então cai na classe negada igual espaço e pontuação. Juntando essa técnica com `remover_acentos` (de cima), dá pra transformar qualquer texto bagunçado num identificador limpo — é exatamente o que o exercício de hoje pede.

## 4. Funções para detectar padrões da língua

Lembrando: uma função em Python recebe entrada e devolve uma saída (`return`). Combinando **slicing**/`.endswith()` com regex, dá pra escrever pequenos "detectores" de características do português. Vamos testar cinco — e depois quebrar cada um de propósito.

In [ ]:
def parece_plural(palavra):
    return palavra.lower().endswith('s')

def parece_diminutivo(palavra):
    return palavra.lower().endswith(('inho', 'inha', 'zinho', 'zinha'))

def parece_infinitivo(palavra):
    return palavra.lower().endswith(('ar', 'er', 'ir')) and len(palavra) > 3

def parece_voz_passiva(frase):
    padrao = r'\b(é|foi|são|foram|será|serão)\s+\w+(ado|ada|ados|adas|ido|ida|idos|idas)\b|\b\w+-se\b'
    return bool(re.search(padrao, frase.lower()))

def contem_linguagem_inadequada(frase, lista_proibida):
    tokens = frase.lower().split()
    return any(t.strip('.,!?') in lista_proibida for t in tokens)


# Casos em que funcionam bem
lista_proibida = {"idiota", "burro", "imbecil"}

print("parece_plural('gatos'):", parece_plural("gatos"))
print("parece_diminutivo('cachorrinho'):", parece_diminutivo("cachorrinho"))
print("parece_infinitivo('estudar'):", parece_infinitivo("estudar"))
print("parece_voz_passiva('A carta foi enviada pelo aluno.'):", parece_voz_passiva("A carta foi enviada pelo aluno."))
print("parece_voz_passiva('Vende-se um carro usado.'):", parece_voz_passiva("Vende-se um carro usado."))
print("contem_linguagem_inadequada('que resposta idiota'):", contem_linguagem_inadequada("que resposta idiota", lista_proibida))

Agora, de propósito, os casos em que essas regras simples quebram — porque a língua portuguesa está cheia de exceção:

In [ ]:
# Quebrando de propósito
print("parece_plural('lápis'):", parece_plural("lápis"))       # é singular, mas termina em 's'
print("parece_plural('ônibus'):", parece_plural("ônibus"))     # idem
print("parece_infinitivo('lugar'):", parece_infinitivo("lugar"))   # substantivo, não verbo — mas tem mais de 3 letras e passa
print("parece_infinitivo('dar'):", parece_infinitivo("dar"))       # verbo de verdade, mas só tem 3 letras — o len > 3 barra ele também
print("contem_linguagem_inadequada('Vimos um burro na fazenda.'):", contem_linguagem_inadequada("Vimos um burro na fazenda.", lista_proibida))  # "burro" aqui é o animal, não ofensa

Todas quebraram, como esperado — inclusive `contem_linguagem_inadequada`, que não tem ideia de que "burro" pode ser bicho de fazenda, não xingamento. Regra escrita à mão cobre o caso comum e erra no caso raro — é exatamente por isso que, mais pra frente no curso, trocamos regra por estatística e aprendizagem de máquina. A próxima seção mostra o que acontece quando esse tipo de detector precisa lidar com texto real.

### Consertando (até onde dá) o detector de plural

Um dos cinco dá pra melhorar de verdade — os outros quatro não, porque a ambiguidade está na própria frase (contexto), não só na palavra. Já "gatos" e "lápis" terminam exatamente do mesmo jeito (um "-s" no fim), então refinar a regra de sufixo não resolve. Mas "lápis", "ônibus", "tênis", "vírus" são um grupo fechado e conhecido da língua — substantivos que não mudam do singular pro plural. Uma lista de exceções resolve os casos conhecidos:

In [ ]:
singulares_terminados_em_s = {"ônibus", "lápis", "tênis", "vírus", "país", "atlas", "pires"}

def parece_plural_v2(palavra):
    p = palavra.lower()
    if p in singulares_terminados_em_s:
        return False
    return p.endswith('s')


for palavra in ["gatos", "lápis", "ônibus", "país", "campus", "bônus"]:
    print(palavra, "->", parece_plural_v2(palavra))

As quatro primeiras acertam agora — inclusive "país", que quebrava por um motivo um pouco diferente (é oxítona acentuada, não "invariável", mas também é singular terminado em "s"). Mas "campus" e "bônus" continuam quebrando: são da mesma categoria, só que não entraram na lista.

**A língua não é uma lista finita.** Dá pra ir aumentando essa lista, mas nunca esgota — português tem gírias, neologismos, empréstimos de outras línguas. Um dicionário que tentasse cobrir *toda* exceção teria que ser gigante, e mesmo assim ficaria incompleto no dia em que alguém criasse uma palavra nova.

**Spoiler de onde o curso vai:** a solução real não é escrever uma lista maior — é trocar "lista escrita à mão" por "padrão aprendido de dado real" (estatística, redes neurais — sessões 5 e 14). Um modelo treinado num corpus grande de português aprende sozinho que "lápis" é invariável, só de ver muitas frases reais — e ainda usa o *contexto* da frase (concordância verbal: "os lápis estão" vs. "o lápis está") pra resolver casos que nenhuma lista resolveria. Isso não é "só LLM" — é a família inteira de modelos estatísticos; LLM é só o membro mais recente dela.

## 5. Caso real: comentários ofensivos de verdade

O detector `contem_linguagem_inadequada` da seção anterior só reconhece as três palavras que a gente decidiu colocar na lista, e não entende contexto nenhum. Sistemas de moderação de verdade não são construídos numa lista escrita à mão — são treinados em dado real, anotado por gente que leu e classificou cada comentário. Vamos ver como é esse dado de verdade, em três níveis: um exemplo sintético do dia a dia da faculdade, uma amostra real filtrada, e uma amostra real sem filtro nenhum.

In [ ]:
# Nível 1 — sintético, do dia a dia da faculdade
comentarios_forum = [
    "Esse professor é super atencioso, recomendo muito a matéria.",
    "A prova foi um lixo, não tinha nada a ver com o que foi dado em aula.",
    "Que preguiça desse trabalho em grupo, ninguém ajuda em nada.",
    "Fulano é um idiota, nunca responde e-mail da monitoria.",
]

for c in comentarios_forum:
    print(f"{contem_linguagem_inadequada(c, lista_proibida)!s:5} — {c}")

Isso funciona pros exemplos que a gente já escreveu pensando na lista de palavras proibidas. Agora vamos ver dado real: o **HateBR**, um corpus de comentários reais do Instagram em português, anotado por especialistas para detecção de discurso de ódio e linguagem ofensiva (VARGAS et al., 2022) — 7.000 comentários, licença CC BY-NC 4.0 (uso não comercial, o que cobre uso didático).

**Importante:** o HateBR é um *dataset* (dado real rotulado), não um modelo — não vamos treinar nem usar nenhum modelo aqui. A ideia é só carregar frases reais com rótulo humano real, e comparar com o nosso detector de regra fixa (`contem_linguagem_inadequada`, com a mesma `lista_proibida` da seção 4). Modelo de verdade — treinado nesse tipo de dado — só entra mais pra frente no curso (sessões 12-13 e 14).

Em vez de digitar exemplo por exemplo, vamos carregar o CSV real direto do repositório com `pandas` — assim o link vira código de verdade, não só uma citação:

- **`anotator1`, `anotator2`, `anotator3`** — o voto individual de cada um dos três leitores humanos que classificaram aquele comentário: `1` se acharam ofensivo, `0` se não.
- **`label_final`** — o rótulo final do comentário (quando pelo menos dois dos três concordam).

In [ ]:
import pandas as pd

url_hatebr = "https://raw.githubusercontent.com/franciellevargas/HateBR/main/dataset/HateBR.csv"
hatebr = pd.read_csv(url_hatebr)  # 7.000 comentários reais, carregados do GitHub

print(hatebr.shape)
hatebr[["id", "comentario", "anotator1", "anotator2", "anotator3", "label_final"]]

Essas são as 7.000 linhas reais, sem seleção nenhuma. Pra caber no notebook, vamos rodar o nosso detector só em 5 delas — escolhidas à mão pelo `id` (não pelo conteúdo), só pra não imprimir as 7.000:

In [ ]:
# Nível 2 — real, filtrado: 5 linhas do dataframe carregado acima, selecionadas só pelo id
comentarios_hatebr_leve = hatebr[hatebr["id"].isin([1, 2, 3, 8, 6999])]
comentarios_hatebr_leve



In [ ]:
for _, linha in comentarios_hatebr_leve.iterrows():
    detectado = contem_linguagem_inadequada(linha["comentario"], lista_proibida)
    anotadores = [linha["anotator1"], linha["anotator2"], linha["anotator3"]]
    print(f"detector ingênuo: {detectado!s:5} | rótulo humano: {linha['label_final']} | anotadores: {anotadores} | {linha['comentario']!r}")

Repare três coisas. Primeiro, o detector ingênuo não pega **nenhum** dos quatro comentários realmente ofensivos — porque nenhuma das três palavras da lista aparece neles; dá pra ser ofensivo sem usar palavrão nenhum, só com ironia ou ataque pessoal. Segundo, olhe os anotadores de "Se elegeu as nossas custas": um dos três discordou (`0`, enquanto os outros dois marcaram `1`). Isso é normal em dado anotado por gente — mesmo humano, às vezes, discorda se algo é ofensivo ou não. Terceiro: o quinto exemplo (`id 6999`) é um comentário elogioso de verdade, com rótulo `0` — aí o `False` do detector está certo, não é falha nenhuma. Um bom detector também precisa acertar quando não tem nada de errado, não só quando tem.

In [ ]:
# Nível 3 — real, sem curadoria: mais uma linha do mesmo dataframe, sem escolher pelas mais "leves"
linha_bruta = hatebr[hatebr["id"] == 10].iloc[0]

print(linha_bruta["comentario"])
print("detector ingênuo:", contem_linguagem_inadequada(linha_bruta["comentario"], lista_proibida))
print("rótulo humano:", linha_bruta["label_final"])

O HateBR completo tem 7.000 comentários e vai bem mais longe do que esses cinco exemplos — inclui categorias de discurso de ódio explícito (racismo, homofobia, xenofobia, entre outras). Não vamos abrir esse nível aqui; o ponto já está feito: moderação de conteúdo de verdade não dá pra resolver com uma lista de palavras proibidas.

**Como esse mesmo dataset seria usado de verdade, mais pra frente no curso:** os 7.000 pares (comentário, rótulo humano) que a gente carregou aqui são exatamente o tipo de dado que serve pra **treinar um modelo de machine learning**. A ideia, resumida: transformar cada comentário em números (sessão 5 — Bag of Words, TF-IDF, embeddings) e usar um algoritmo (ex.: Naive Bayes, SVM — sessão 13) que aprende sozinho, olhando milhares desses exemplos, que padrão de palavras/contexto costuma aparecer em comentário ofensivo. Depois de treinado, esse modelo classifica comentário **novo, nunca visto**, sem ninguém escrever regra nenhuma pra aquele caso — é a forma real de resolver o problema que a lista fixa (`contem_linguagem_inadequada`) não resolve.

E antes de seguir pro spaCy: vale testar o detector de diminutivo (`parece_diminutivo`, lá da seção 4) num contexto bem familiar pra vocês:

In [ ]:
comentarios_faculdade = [
    "Só faltou terminar aquele trabalhinho de última hora.",
    "Passei a madrugada estudando pra provinha de amanhã.",
    "Já mandei um e-mail pro professorzinho novo, vamos ver se ele responde.",
    "Semana que vem é só resuminho e exercício, tranquilo.",
]

for c in comentarios_faculdade:
    diminutivos = [t.strip('.,!?') for t in c.split() if parece_diminutivo(t.strip('.,!?'))]
    print(f"{c!r} → diminutivos: {diminutivos}")

Repare que nem todo diminutivo é sobre tamanho — "provinha" e "trabalhinho" não são versões pequenas de prova/trabalho, é tom informal/afetivo (às vezes até irônico, quando a "provinha" é enorme). `parece_diminutivo` não distingue isso, só olha o sufixo — mais uma regra que cobre o caso comum e ignora a intenção por trás da palavra.

## 6. NLTK vs spaCy

Até aqui usamos só o NLTK. Vamos rodar a mesma tokenização com o spaCy e comparar.

In [ ]:
import spacy

try:
    nlp = spacy.load("pt_core_news_sm")
except OSError:
    # primeira vez rodando nesta máquina: instala o spaCy e baixa o modelo de português
    # (o modelo tem uns 13MB, isso só roda uma vez)
    !pip install spacy
    !python -m spacy download pt_core_news_sm
    nlp = spacy.load("pt_core_news_sm")

In [ ]:
# Mesma frase difícil da seção 1, agora tokenizada pelo spaCy
doc = nlp(frase_dificil)
tokens_spacy = [token.text for token in doc]

print("NLTK: ", word_tokenize(frase_dificil, language='portuguese'))
print("spaCy:", tokens_spacy)

Repare como o spaCy lida com a URL e com "guarda-chuva" — igual, diferente ou melhor que o NLTK? E as stopwords, também vêm prontas no spaCy:

In [ ]:
from nltk.corpus import stopwords

stopwords_nltk = set(stopwords.words('portuguese'))
stopwords_spacy = nlp.Defaults.stop_words

print(f"NLTK:  {len(stopwords_nltk)} stopwords")
print(f"spaCy: {len(stopwords_spacy)} stopwords")

print("Só no NLTK (exemplos):", list(stopwords_nltk - stopwords_spacy)[:20])
print("Só no spaCy (exemplos):", list(stopwords_spacy - stopwords_nltk)[:20])

In [ ]:
# 2. INTERSEÇÃO (&): Quais palavras existem em AMBOS (comum acordo entre as bibliotecas)
palavras_em_ambos = stopwords_nltk & stopwords_spacy
print(f"3. Presentes em ambos ({len(palavras_em_ambos)} palavras). Exemplos:")
print(list(palavras_em_ambos)[:10], "\n")

# 3. DIFERENÇA SIMÉTRICA (^): Quais palavras são exclusivas (estão em um OU no outro, mas não em ambos)
exclusivas = stopwords_nltk ^ stopwords_spacy
print(f"4. Palavras exclusivas de biblioteca ({len(exclusivas)} palavras). Exemplos:")
print(list(exclusivas)[:10], "\n")

# 4. UNIÃO (|): Juntar as duas listas sem repetir palavras
todas_stopwords = stopwords_nltk | stopwords_spacy
print(f"5. Total de stopwords únicas se juntarmos as duas bibliotecas: {len(todas_stopwords)}")

In [ ]:
'então' in stopwords_nltk

In [ ]:
'então' in stopwords_spacy

In [ ]:
# Pega o que tem no NLTK, tirando tudo que também tem no spaCy
exclusivas_nltk = stopwords_nltk - stopwords_spacy

# Pega o que tem no spaCy, tirando tudo que também tem no NLTK
exclusivas_spacy = stopwords_spacy - stopwords_nltk

# Mostra a quantidade total e alguns exemplos
print(f"Exclusivas SÓ do NLTK ({len(exclusivas_nltk)} palavras). Exemplos:")
print(list(exclusivas_nltk)[:15], "\n")

print(f"Exclusivas SÓ do spaCy ({len(exclusivas_spacy)} palavras). Exemplos:")
print(list(exclusivas_spacy)[:15])

## 7. Stopwords

Agora o segundo problema da sessão passada: palavras como "o", "e", "a" dominando a lista de frequência.

In [ ]:
from nltk.corpus import stopwords

stopwords_pt = stopwords.words('portuguese')
print(f"Total de stopwords em português: {len(stopwords_pt)}")
print(stopwords_pt[:15])

Essa lista não é fixa — dá pra ajustar pro seu caso, com operação de conjunto (`set`):

In [ ]:
stop_pt_set = set(stopwords_pt)

# Adicionar palavra própria da sua tarefa (ex.: termos genéricos demais numa avaliação de produto)
stop_pt_customizada = stop_pt_set | {"pra", "tá", "aí", "app"}
print("Adicionando:", "pra" in stop_pt_set, "->", "pra" in stop_pt_customizada)

# Tirar uma palavra que você quer manter (ex.: "não", pra não perder a negação)
stop_pt_sem_negacao = stop_pt_set - {"não"}
print("Removendo:", "não" in stop_pt_set, "->", "não" in stop_pt_sem_negacao)

In [ ]:
texto_avaliacao = (
    "Comprei esse fone de ouvido semana passada e, sinceramente, não esperava tanto. "
    "A bateria dura o dia inteiro, o som é ótimo pra esse preço e o app conectou super rápido. "
    "Só achei a caixinha um pouco frágil, mas isso não estragou minha experiência. Recomendo!"
)

tokens = word_tokenize(texto_avaliacao, language='portuguese')
tokens = [t.lower() for t in tokens if t.isalpha()]
tokens_sem_stopwords = [t for t in tokens if t not in stopwords_pt]

print("Sem stopwords:", tokens_sem_stopwords)

**Pergunta pra pensar (responda na célula de markdown no fim):** esse é o mesmo texto do fone de ouvido da sessão 2. Você usou "não" duas vezes nesse texto ("não esperava", "não estragou") — o que aconteceu com o "não" na lista acima? Isso seria um problema se a tarefa fosse análise de sentimento?

## 8. Juntando tudo num pipeline

Uma função que faz as quatro etapas de uma vez — normalizar, tokenizar, filtrar pontuação, remover stopwords (opcional).

In [ ]:
def limpar_texto(texto, remover_stopwords=True):
    tokens = word_tokenize(texto.lower(), language='portuguese')
    tokens = [t for t in tokens if t.isalpha()]
    if remover_stopwords:
        tokens = [t for t in tokens if t not in stopwords_pt]
    return tokens


tokens_limpos = limpar_texto(texto_avaliacao)
frequencia = Counter(tokens_limpos)

print("Frequência depois da limpeza completa:")
for palavra, contagem in frequencia.most_common(10):
    print(f"{palavra!r}: {contagem}")

Compare essa lista com a da sessão 2 (que tinha vírgula, ponto, "o", "e", "a" no topo). Agora o que sobra já começa a dizer algo sobre o assunto do texto — isso é a base de tudo que vem depois no curso, principalmente extração de características (sessão 5).

Resolvemos pontuação e stopwords — mas mesmo depois de tudo minúsculo e limpo, "estudar", "estudando" e "estudei" continuam sendo tokens diferentes pro computador, embora carreguem a mesma ideia central. É esse o assunto da Parte II.

## 9. Stemming

Corta o sufixo da palavra por regra, sem se importar se o resultado é uma palavra real — reduz formas parecidas a uma raiz aproximada.

In [ ]:
from nltk.stem import RSLPStemmer

nltk.download('rslp')  # RSLP = Removedor de Sufixos da Língua Portuguesa

stemmer = RSLPStemmer()

for palavra in ["estudar", "estudando", "estudei", "estudo", "estudantil"]:
    print(palavra, "->", stemmer.stem(palavra))

Rápido e barato, mas grosseiro — o resultado às vezes nem é uma palavra real. E só dá pra fazer isso com NLTK: o spaCy não tem stemmer (decisão de projeto — a equipe considera que lematização já resolve melhor o mesmo problema). É o contrário na próxima seção.

## 10. Lematização

Reduz a palavra à sua forma de dicionário (o lema) — usa a morfologia real da língua e a classe gramatical, não corta às cegas. O NLTK não tem lematizador de português confiável (o `WordNetLemmatizer` dele depende do WordNet, majoritariamente em inglês) — aqui só dá pra usar o spaCy.

In [ ]:
doc = nlp("estudar estudando estudei estudo estudantil")

for token in doc:
    print(token.text, "->", token.lemma_, f"({token.pos_})")

Repare: só "estudando" virou "estudar". "estudei" devia virar "estudar" e não virou — o modelo pequeno (`pt_core_news_sm`) não tem essa forma no dicionário de lemas. E "estudo" ficou "estudo" por um motivo bom: aqui foi reconhecido como substantivo (repare o `NOUN` ao lado), não verbo — "o estudo", não "eu estudo". Isso já é mais inteligente que o stemming, mesmo errando.

## 11. Stemming vs. lematização, lado a lado

Testando mais palavras nos dois ao mesmo tempo:

In [ ]:
palavras_teste = ["estudar", "estudando", "estudei", "estudo", "estudantil", "casas", "gatinhos"]

print(f"{'palavra':15}{'stemming':12}lematização")
for p in palavras_teste:
    stem = stemmer.stem(p)
    lema = nlp(p)[0].lemma_
    print(f"{p:15}{stem:12}{lema}")

Repare "casas" -> "cas" (stemming, não é palavra) vs. "casa" (lematização, palavra real) — e "gatinhos" -> "gat" vs. "gatinho". Nenhuma das duas é sempre a escolha certa — depende de quanto erro grosseiro a tarefa tolera.

## 12. Correção ortográfica

Texto escrito por gente de verdade vem com erro de digitação. A forma mais simples possível de detectar isso — a raiz de boa parte do PLN, antes de qualquer sofisticação — é um **vocabulário**: uma lista de palavras que você considera "corretas". Se a palavra que está sendo checada não está nessa lista, ela é sinalizada como possível erro.

In [ ]:
vocabulario_pt = {
    "casa", "gato", "cachorro", "estudar", "estudante", "professor", "aula", "prova",
    "trabalho", "computador", "internet", "aplicativo", "celular", "livro", "caderno",
    "caneta", "mesa", "cadeira", "janela", "porta", "rua", "cidade", "estado", "país",
    "comida", "água", "café", "leite", "pão", "fruta", "banana", "maçã", "laranja",
    "carro", "ônibus", "avião", "viagem", "praia", "montanha", "floresta", "rio",
    "amigo", "família", "irmão", "irmã", "mãe", "pai", "filho", "filha", "dinheiro",
}
print("Tamanho do vocabulário:", len(vocabulario_pt))


def parece_erro_ortografico(palavra, vocabulario):
    return palavra.lower() not in vocabulario


for palavra in ["casa", "cas", "professora", "computador", "computadr"]:
    print(palavra, "->", "possível erro" if parece_erro_ortografico(palavra, vocabulario_pt) else "ok")

Com só 50 palavras no vocabulário, "cas" e "computadr" são sinalizados certo — mas "professora" *também* vira "possível erro", mesmo sendo uma palavra perfeitamente correta. Não é bug: "professora" simplesmente não está nas 50 palavras que colocamos na lista (só "professor" está). Pra esse método funcionar de verdade, o vocabulário precisaria de *todas* as formas de *todas* as palavras da língua — plural, feminino, cada conjugação de cada verbo — e essa é, de novo, a mesma lição da seção anterior: a língua não é uma lista finita, por maior que a lista seja.

Além disso, esse método só responde "está no vocabulário" ou "não está" — não sugere nenhuma correção. É aí que a distância de edição melhora a ideia: em vez de só "está no dicionário?", ela pergunta "qual palavra do dicionário está mais perto?".

A ideia por trás da correção automática mais comum é a **distância de edição** (Levenshtein): contar quantas operações — inserir, remover ou trocar uma letra — transformam a palavra errada na mais próxima de um dicionário.

In [ ]:
try:
    from spellchecker import SpellChecker
except ModuleNotFoundError:
    # primeira vez rodando nesta máquina: instala o pyspellchecker
    !pip install pyspellchecker
    from spellchecker import SpellChecker

corretor = SpellChecker(language='pt')

for palavra in ["linguajem", "computadr", "trabalho"]:  # a última já está certa, de propósito
    print(palavra, "->", corretor.correction(palavra))

**Cuidado:** correção automática pode "corrigir" gíria, nome próprio ou termo técnico pra algo errado — repare que "trabalho" (já certo) não foi alterado, mas vale sempre checar o que o corretor mudou antes de aplicar automático num pipeline.

## 13. Atualizando o pipeline

Juntando tudo: uma versão nova do `limpar_texto`, que aceita reduzir por stemming ou lematização.

In [ ]:
def limpar_texto_v2(texto, remover_stopwords=True, reduzir=None):
    # Tokeniza e lematiza (se pedido) com o spaCy, sobre a frase original —
    # não sobre uma lista de tokens já filtrada, senão o modelo perde o contexto
    doc = nlp(texto.lower())
    tokens = []
    for tok in doc:
        if not tok.is_alpha:
            continue
        if remover_stopwords and tok.is_stop:
            continue
        forma = tok.lemma_ if reduzir == 'lema' else tok.text
        tokens.append(forma)

    if reduzir == 'stem':
        tokens = [stemmer.stem(t) for t in tokens]

    return tokens


print("Nenhum:      ", limpar_texto_v2(texto_avaliacao, reduzir=None))
print("Stemming:    ", limpar_texto_v2(texto_avaliacao, reduzir='stem'))
print("Lematização: ", limpar_texto_v2(texto_avaliacao, reduzir='lema'))

Repare a ordem: lematizamos sobre a frase original inteira, com o spaCy fazendo tokenização e lematização juntas — não sobre uma lista de tokens já cortada de pontuação/stopwords. Se lematizasse depois de já ter filtrado, o spaCy perderia o contexto da frase e erraria muito mais (é a mesma lição de sempre: ordem importa). Mesmo assim, repare "ótimo" virando "bom" na lematização — outro lembrete de que nem o modelo pequeno é perfeito.